Training of mentions extractor:

In [ ]:
import mentions_extractor_runner

runner = mentions_extractor_runner.MentionsExtractorRunner()
runner.train()

Test to improve mentions linker:

In [ ]:
import torch
import ffnn
k = 5
emb_size = 2

top_spans_mention_score = torch.rand(k)
top_spans_embeddings = torch.rand([k, emb_size])

# create a matrix containing the antecedents' indices of the top scoring spans
antecedents_distance = torch.tile(torch.arange(0, k) + 1, [k, 1])  # [k ,k]
print("Antecedents indices:\n" + str(antecedents_distance))
print("Antecedents indices size:\n" + str(antecedents_distance.size()))

raw_antecedents = torch.unsqueeze(torch.arange(k), 1) - antecedents_distance  # [k, k]
print("Raw antecedents:\n" + str(raw_antecedents))
print("Raw antecedents size:\n" + str(raw_antecedents.size()))

# create a mask for out of bound indices
antecedents_mask = raw_antecedents >= 0  # [k, k]
print("Raw antecedents mask:\n" + str(antecedents_mask))
print("Raw antecedents size:\n" + str(antecedents_mask.size()))

antecedents = torch.maximum(raw_antecedents, torch.tensor(0))  # [k, k]
print("Antecedents:\n" + str(antecedents))
print("Antecedents size:\n" + str(antecedents.size()))

pairs_mention_score = top_spans_mention_score.unsqueeze(1) + top_spans_mention_score[antecedents]  # [k, k]
# use mask to put out of bound values to -inf
pairs_mention_score += torch.log(antecedents_mask.float())  # [k, k]
print("Pairs mention score:\n" + str(pairs_mention_score))
print("Pairs mention score size:\n" + str(pairs_mention_score.size()))

antecedents_emb = top_spans_embeddings[antecedents] # [k, k, emb]
print("Antecendent emb size:\n" + str(antecedents_emb.size()))

target_embeddings = top_spans_embeddings.unsqueeze(1).tile([1, k, 1])  # [k, 1, emb]
print("Target emb size:\n" + str(target_embeddings.size()))
mult_embeddings = target_embeddings * top_spans_embeddings  # [k, k, emb]
print("Multiplication emb size:\n" + str(mult_embeddings.size()))
pairs_embeddings = torch.cat([target_embeddings, antecedents_emb, mult_embeddings], 2)  # [k, k, 3*emb]
print("Pairs emb size:\n" + str(pairs_embeddings.size()))

ffnn = ffnn.FFNN(3 * emb_size)
pairs_antecedent_score = ffnn(pairs_embeddings).squeeze(2)  # [k, k]
print("Pairs antecedent score size: " + str(pairs_antecedent_score.size()))

pairs_score = pairs_mention_score + pairs_antecedent_score


Forward pass in linker:

In [1]:
import torch
import utils
from transformers import AutoModel, AutoTokenizer
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained("SpanBERT/spanbert-base-cased").to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
sampleText = "Hello, my name is Damien. It's a cool name I think!"
tokenized_text = tokenizer.encode_plus(sampleText, return_tensors='pt')
tokenized_text["input_ids"] = tokenized_text["input_ids"].squeeze()
tokenized_text["token_type_ids"] = tokenized_text["token_type_ids"].squeeze()
tokenized_text["attention_mask"] = tokenized_text["attention_mask"].squeeze()
encoded_text = utils.spanbert_encode(tokenized_text, encoder)

Some weights of BertModel were not initialized from the model checkpoint at SpanBERT/spanbert-base-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
import mentions_extractor
import mentions_linker

extractor = mentions_extractor.MentionsExtractor(768, 0.25, 10).to(device)
linker = mentions_linker.MentionsLinker(extractor, 768).to(device)

res = linker(encoded_text)
print(res)

tensor([[  -inf,   -inf,   -inf,   -inf],
        [0.0532,   -inf,   -inf,   -inf],
        [0.0494, 0.0489,   -inf,   -inf],
        [0.0411, 0.0449, 0.0450,   -inf]], device='cuda:0',
       grad_fn=<AddBackward0>)
